<a href="https://colab.research.google.com/github/minxuan0923/AML_Assignment/blob/mx/Set5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import libraries

In [50]:
import pandas as pd
from scipy.io import arff
import requests
from io import StringIO

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score

import tensorflow as tf
from tensorflow.keras import datasets, layers, models, optimizers, losses
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

## Load Dataset

In [51]:
URL = (
    "https://raw.githubusercontent.com/"
    "minxuan0923/AML_Assignment/main/data/"
    "UCI_Phishing_Training_Dataset.arff"
)

response = requests.get(URL, timeout=30)
response.raise_for_status()

data, meta = arff.loadarff(StringIO(response.text))

df = pd.DataFrame(data)

for col in df.select_dtypes([object]):
    df[col] = pd.to_numeric(df[col].str.decode('utf-8'))

X_data = df.drop(columns =['Result'])
y_data = df['Result']

## Spliting

In [52]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_data, y_data, random_state = 42, test_size=0.3, stratify=y_data)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


## Encoding

In [53]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)

# One-hot-encoding X values
X_train_encoded = encoder.fit_transform(X_train)
X_val_encoded = encoder.fit_transform(X_val)
X_test_encoded = encoder.transform(X_test)

# Convert y values - phishing to "1" and legitimate to "0"
y_train = y_train.replace({-1: 1, 1: 0})
y_val = y_val.replace({-1: 1, 1: 0})
y_test = y_test.replace({-1: 1, 1: 0})

## Autoencoder

In [55]:

class Autoencoder(Model):
  def __init__(self, latent_dim, shape):
    super(Autoencoder, self).__init__()
    self.latent_dim = latent_dim
    self.shape = shape
    self.encoder = tf.keras.Sequential([
      layers.Flatten(),
      layers.Dense(latent_dim, activation='relu'),
    ])
    self.decoder = tf.keras.Sequential([
      layers.Dense(int(tf.math.reduce_prod(shape)), activation='sigmoid'),
      layers.Reshape(shape)
    ])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded


shape = X_test_encoded.shape[1:]
latent_dim = 16 # we will experiment with 4, 8, 16, 32
autoencoder = Autoencoder(latent_dim, shape)

autoencoder.compile(optimizer='adam', loss=losses.binary_crossentropy)

history_ae = autoencoder.fit(X_train_encoded, X_train_encoded,
                epochs=10,
                shuffle=True,
                validation_data=(X_val_encoded, X_val_encoded))

X_train_compressed = autoencoder.encoder.predict(X_train_encoded)
X_val_compressed = autoencoder.encoder.predict(X_val_encoded)
X_test_compressed = autoencoder.encoder.predict(X_test_encoded)

Epoch 1/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.4788 - val_loss: 0.3458
Epoch 2/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2909 - val_loss: 0.2483
Epoch 3/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2130 - val_loss: 0.1830
Epoch 4/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.1621 - val_loss: 0.1451
Epoch 5/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.1311 - val_loss: 0.1200
Epoch 6/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.1100 - val_loss: 0.1030
Epoch 7/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0955 - val_loss: 0.0911
Epoch 8/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0856 - val_loss: 0.0831
Epoch 9/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0788 - val_loss: 0.0773
Epoch 10/10
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0739 - val_loss: 0.0738
242/242 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


## Logistic Regression

In [56]:
# Gradient descent
logistic_model = models.Sequential()
logistic_model.add(layers.Normalization(input_shape = [16,], axis = None))
logistic_model.add(layers.Dense(1, activation = 'sigmoid'))

logistic_model.compile(optimizer='adam',
              loss=losses.binary_crossentropy,
              metrics=['accuracy'])

logistic_model.summary()

history_lr = logistic_model.fit(X_train_compressed, y_train, epochs=50,
                    validation_data=(X_val_compressed, y_val))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_3 (Normalization) │ (None, 16)             │             3 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20 (84.00 B)

 Trainable params: 17 (68.00 B)

 Non-trainable params: 3 (16.00 B)

Epoch 1/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4661 - loss: 3.9877 - val_accuracy: 0.3866 - val_loss: 2.0773
Epoch 2/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4178 - loss: 1.7986 - val_accuracy: 0.4741 - val_loss: 1.4058
Epoch 3/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5277 - loss: 1.1804 - val_accuracy: 0.6074 - val_loss: 0.8805
Epoch 4/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6639 - loss: 0.7522 - val_accuracy: 0.7461 - val_loss: 0.5684
Epoch 5/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7786 - loss: 0.5084 - val_accuracy: 0.8221 - val_loss: 0.4053
Epoch 6/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8488 - loss: 0.3790 - val_accuracy: 0.8860 - val_loss: 0.3161
Epoch 7/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8864 - loss: 0.3090 - val_accuracy: 0.9083 - val_loss: 0.2694
Epoch 8/50
242/242 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9017 - loss: 0.2671 - val_accuracy: 0.

## Evaluate Model

In [58]:
def evaluate_model(model, X, y):
    y_prob = model.predict(X, verbose=0)
    y_pred = (y_prob >= 0.5).astype(int).flatten()
    results = {
        'Accuracy': accuracy_score(y, y_pred),
        'Precision': precision_score(y, y_pred, zero_division=0),
        'Recall': recall_score(y, y_pred, zero_division=0),
        'F1-score': f1_score(y, y_pred, zero_division=0),
        'F2-score': fbeta_score(y, y_pred, beta=2),
    }

    return results



results = evaluate_model(

    logistic_model,
    X_val_compressed,
    y_val
)

print(results)

{'Accuracy': 0.9348612786489746, 'Precision': 0.9459459459459459, 'Recall': 0.9047619047619048, 'F1-score': 0.9248956884561892, 'F2-score': 0.9127093055174307}
